# V0.9B Memory Correctness Lab

问题：已经记住的信息如果过期、冲突、或与当前观察不一致，Kernel 应该怎么办？

术语：

- Memory：长期记住的语义信息，不等于客观真相。
- Current Observation：本轮通过工具看到的当前证据，例如读取 `pyproject.toml`。
- STALE：记忆仍在 durable history 里，但默认不再进入检索/上下文。
- SUPERSEDED：旧记忆被明确的新 Memory 替代。
- Conflict：两条 Memory 显式冲突，但这不是 store 损坏。
- Provenance：说明这条记忆或 lifecycle 变化从哪里来。

本 lab 是 deterministic/offline；没有调用真实模型，也没有自动判真。

In [ ]:
from pathlib import Path
import sys
from tempfile import TemporaryDirectory

repo = Path.cwd()
if not (repo / "agentkernel").exists():
    repo = repo.parent
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

from agentkernel import (
    CapabilityEvaluator,
    CapabilityGrant,
    JsonlMemoryStore,
    MEMORY_READ_ACTION,
    MEMORY_WRITE_ACTION,
    MemoryProvenance,
    MemoryService,
    memory_namespace_scope,
    project_conflicting_memories_to_context_pages,
    project_memories_to_context_pages,
)

AGENT = "lab-agent"
NAMESPACE = "project"
tmp = TemporaryDirectory()
store_path = Path(tmp.name) / "memory.jsonl"

def grants(*actions):
    return CapabilityEvaluator(
        CapabilityGrant(AGENT, action, memory_namespace_scope(AGENT, NAMESPACE))
        for action in actions
    )

def provenance(source="host", **kwargs):
    return MemoryProvenance(source=source, source_agent_id=AGENT, **kwargs)

def show(title, lines):
    print(f"\n=== {title} ===")
    if isinstance(lines, dict):
        for key, value in lines.items():
            print(f"{key}: {value}")
    else:
        print(lines)

memory = MemoryService(JsonlMemoryStore(store_path))
show("Setup", {"agent": AGENT, "namespace": NAMESPACE, "store": store_path})

## Story A: Durable != Fresh

Session A 记住：项目使用 Python 3.11。随后 Session B 通过当前文件观察到：项目要求 Python >=3.12。Kernel 不自动判真，只允许上层显式把旧 Memory 标记为 STALE。

In [ ]:
m1 = memory.remember(
    agent_id=AGENT,
    namespace=NAMESPACE,
    content="Project requires Python 3.11.",
    provenance=provenance("session", source_session_id="session-A", source_event_id="user-1"),
    capability_evaluator=grants(MEMORY_WRITE_ACTION),
)

current_observation = {
    "source": "pyproject.toml",
    "observed": "requires-python >=3.12",
    "session": "session-B",
    "tool_result": "read-file-1",
}

show("Remembered vs Current Observation", {
    "memory": m1.content,
    "memory_id": m1.memory_id,
    "current_observation": current_observation,
    "kernel_decision": "no automatic truth decision",
})

In [ ]:
stale = memory.mark_stale(
    m1.memory_id,
    agent_id=AGENT,
    reason="Current pyproject.toml requires Python >=3.12.",
    evidence_provenance=provenance(
        "current-observation",
        source_session_id="session-B",
        source_event_id="read-file-1",
        source_tool_name="read_file",
        note="pyproject.toml requires-python >=3.12",
    ),
    capability_evaluator=grants(MEMORY_WRITE_ACTION),
)

default_search = memory.search(
    agent_id=AGENT,
    owner_agent_id=AGENT,
    namespace=NAMESPACE,
    query="Python",
    limit=5,
    capability_evaluator=grants(MEMORY_READ_ACTION),
)
history = memory.history(
    agent_id=AGENT,
    owner_agent_id=AGENT,
    namespace=NAMESPACE,
    capability_evaluator=grants(MEMORY_READ_ACTION),
)

show("After mark_stale", {
    "default_search_count": len(default_search),
    "history_status": history[0].lifecycle_state,
    "stale_reason": history[0].stale_reason,
    "stale_evidence": history[0].stale_provenance.as_dict(),
})

## Restart Check

换一个新的 `MemoryService`，模拟新 runtime。Memory 的 stale metadata 应该从 JSONL durable events 重建。

In [ ]:
memory.close()
memory = MemoryService(JsonlMemoryStore(store_path))
restored = memory.read(
    m1.memory_id,
    agent_id=AGENT,
    include_inactive=True,
    capability_evaluator=grants(MEMORY_READ_ACTION),
)
show("Fresh Runtime Restores Lifecycle", {
    "same_memory_id": restored.memory_id,
    "status": restored.lifecycle_state,
    "stale_reason": restored.stale_reason,
    "evidence_source": restored.stale_provenance.source,
})

## Story B: Conflict != Corruption

现在写入两条 active Memory：偏好 Python 与偏好 Rust。Kernel 不自动选择谁是真的；上层显式建立 conflict relation 后，Context projection 可以把双方一起展示给模型。

In [ ]:
m2 = memory.remember(
    agent_id=AGENT,
    namespace=NAMESPACE,
    content="Preferred implementation language is Python.",
    provenance=provenance("session", source_session_id="session-C", source_event_id="user-2"),
    capability_evaluator=grants(MEMORY_WRITE_ACTION),
)
m3 = memory.remember(
    agent_id=AGENT,
    namespace=NAMESPACE,
    content="Preferred implementation language is Rust.",
    provenance=provenance("session", source_session_id="session-D", source_event_id="user-3"),
    capability_evaluator=grants(MEMORY_WRITE_ACTION),
)
conflicted = memory.mark_conflict(
    agent_id=AGENT,
    memory_ids=(m2.memory_id, m3.memory_id),
    reason="Two remembered language preferences disagree.",
    evidence_provenance=provenance("host", source_session_id="review", source_event_id="review-1"),
    conflict_group_id="conflict_language",
    capability_evaluator=grants(MEMORY_WRITE_ACTION),
)

show("Conflict Relation", {
    item.memory_id: {
        "status": item.lifecycle_state,
        "conflict_group": item.conflict_group_id,
        "conflicts_with": item.conflicts_with_memory_ids,
    }
    for item in conflicted
})

In [ ]:
projection = project_conflicting_memories_to_context_pages(conflicted, top_k=10)
for page in projection.pages:
    print("\n--- Model-visible ContextPage ---")
    print(page.content)

show("Conclusion", {
    "durable_not_fresh": "stale memory survived but default search hides it",
    "conflict_not_corruption": "both conflicting memories remain active and visible for diagnosis",
    "kernel_policy_boundary": "Kernel records relation/provenance; it does not choose truth",
})